# 06 · La completesa de ℝ

[Obre aquest quadern a Google Colab](https://colab.research.google.com/github/mlacasa/1BATXILLERAT/blob/main/06_Completesa_R.ipynb)

**Matemàtiques · 1r de batxillerat**  
**Autoria: Dr. Lacasa-Cazcarra · Any 2026**

**Objectiu.** Connectar suprem, intervals encaixats i successions de Cauchy sense raonaments circulars.

**Abans de començar:** quaderns 01–05  
**Temps orientatiu:** 2 sessions; demostracions avançades opcionals. Hi ha activitats bàsiques i ampliacions opcionals.

**Funcionament.** Executa les cel·les en ordre, des del començament. Primer fes una predicció,
després experimenta i escriu una justificació. Els controls són opcionals: també pots
modificar els arguments de les funcions i executar-les. Les figures representen mostres
finites; les propietats infinites necessiten una demostració.

**Procedència.** Quadern nou. Síntesi dels experiments d'Arquímedes, Dedekind i Cauchy.


In [ ]:
import math
from fractions import Fraction
from decimal import Decimal, localcontext
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "font.size": 11, "figure.dpi": 100})

def explora(funcio, **rangs):
    """Controls opcionals: la mateixa funció també es pot cridar directament."""
    try:
        import ipywidgets as widgets
    except ImportError:
        print("Sense controls: executem els paràmetres per defecte. Pots canviar-los a la crida.")
        return funcio()
    panell = widgets.interactive(funcio, **rangs)
    display(panell)
    return panell


## 1. Màxim, cota superior i suprem
Una **cota superior** de S és un nombre M tal que $s\le M$ per a tot $s\in S$.
El **suprem** és la menor de les cotes superiors. Un **màxim** és un element de S
que és més gran o igual que tots els altres; el suprem no ha de pertànyer a S.

Per a $S=\{1-1/n:n\ge1\}$, 1 és el suprem però no és un màxim.
Qualsevol llista finita dels primers termes sí que té màxim. No confonguis la mostra amb S.


In [ ]:
def suprem_mostra(termes=12):
    n = np.arange(1, termes+1)
    valors = 1-1/n
    fig, ax = plt.subplots()
    ax.plot(n, valors, "o", label="Mostra finita")
    ax.axhline(1, color="#dc2626", label="Suprem del conjunt infinit: 1")
    ax.set(xlabel="n", ylabel="1 − 1/n", title="El màxim de la mostra no és el suprem de S")
    ax.legend(); plt.show()
    print("Màxim exacte de la mostra:", 1-Fraction(1, termes))
panell = explora(suprem_mostra, termes=(2, 100, 1))


## 2. El punt de partida lògic
**Axioma de completesa:** tot subconjunt no buit de ℝ acotat superiorment té suprem en ℝ.

El prenem com a propietat fonamental del sistema dels reals, juntament amb les
propietats de cos ordenat. No el deduïm d'un gràfic ni d'un càlcul finit de π.
La construcció per talladures és una altra manera de donar un model d'aquest sistema.

**Propietat arquimediana.** Els naturals no estan acotats superiorment en ℝ.
En efecte, si tinguessin suprem s, s−1 no seria cota superior: existiria un natural
n>s−1, i llavors n+1>s, contradicció. Això justifica que $2^{-n}$ pot ser tan petit
com vulguem. No confonguis aquesta propietat amb l'algorisme geomètric d'Arquímedes.

## 3. Del suprem als intervals encaixats
Suposem $I_n=[a_n,b_n]$, no buits, tancats, amb $I_{n+1}\subseteq I_n$.
El conjunt dels extrems inferiors és no buit i està acotat per $b_0$.
Sigui $x=\sup\{a_n:n\ge0\}$.

Per a cada n, $a_n\le x$. A més, $b_n$ és cota superior de **tots** els $a_k$:
si $k\le n$, $a_k\le a_n\le b_n$; si $k\ge n$, $a_k\le b_k\le b_n$.
Per tant $x\le b_n$ i $x\in I_n$ per a tot n.
Si $b_n-a_n\to0$, dos punts comuns x,y haurien de satisfer
$|x-y|\le b_n-a_n$ per a tot n; això obliga $x=y$.

**Cal mirar les hipòtesis:** els intervals oberts $(0,1/n)$ són encaixats però no tenen
cap punt comú. Els intervals tancats $[0,1]$ repetits tenen molts punts comuns,
perquè la seva amplada no tendeix a zero.


In [ ]:
def intervals_arrel2(passos=12):
    """Extrems racionals exactes. No calcula sqrt(2)."""
    if not isinstance(passos, int) or passos < 0:
        raise ValueError("El nombre de passos ha de ser un enter no negatiu.")
    a, b = Fraction(1), Fraction(2)
    intervals = [(a, b)]
    for _ in range(passos):
        m = (a + b) / 2
        if m * m < 2:
            a = m
        else:
            b = m
        intervals.append((a, b))
    return intervals


In [ ]:
def arquimedes(passos=8, precisio=60):
    """Semiperímetres calculats amb Decimal; les xifres tenen arrodoniment."""
    if not isinstance(passos, int) or passos < 0:
        raise ValueError("El nombre de passos ha de ser un enter no negatiu.")
    if not isinstance(precisio, int) or precisio < 16:
        raise ValueError("Calen almenys 16 xifres de precisió.")
    with localcontext() as ctx:
        ctx.prec = precisio
        n, inferior, superior = 6, Decimal(3), 2 * Decimal(3).sqrt()
        files = [(n, inferior, superior)]
        for _ in range(passos):
            nova_superior = 2 * inferior * superior / (inferior + superior)
            nova_inferior = (inferior * nova_superior).sqrt()
            n, inferior, superior = 2 * n, nova_inferior, nova_superior
            files.append((n, inferior, superior))
    return files


In [ ]:
def compara_intervals(passos=6):
    racional = intervals_arrel2(passos)
    pi_intervals = arquimedes(passos)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    for k, (a, b) in enumerate(racional):
        axs[0].plot([float(a), float(b)], [k, k], "o-", color="#2563eb")
    for k, (_, a, b) in enumerate(pi_intervals):
        axs[1].plot([float(a), float(b)], [k, k], "o-", color="#d97706")
    axs[0].set_title("Bisecció: el punt comú no pertany a ℚ")
    axs[1].set_title("Arquímedes: el punt comú és π")
    for ax in axs: ax.set(xlabel="Extrems", ylabel="Pas")
    plt.tight_layout(); plt.show()
panell = explora(compara_intervals, passos=(1, 12, 1))


## 4. Per què tota successió de Cauchy real convergeix? — Ampliació
Una successió de Cauchy és acotada. Per a cada N considerem
$$\alpha_N=\inf\{a_n:n\ge N\},\qquad \beta_N=\sup\{a_n:n\ge N\}.$$
El suprem i l'ínfim existeixen per completesa (l'ínfim es dedueix aplicant el
suprem als oposats). Els intervals $[\alpha_N,\beta_N]$ són tancats i encaixats.

Per a cada ε>0, Cauchy dona una cua amb totes les distàncies menors que ε/2;
per tant $\beta_N-\alpha_N\le\varepsilon/2<\varepsilon$.
Les amplades tendeixen a zero. El teorema anterior dona un únic punt L i,
com que tots els termes de la cua són dins l'interval, $a_n\to L$.

**Esquema de les implicacions justificades aquí:**
axioma del suprem → intervals encaixats → convergència de les successions de Cauchy.
En ℝ aquestes són formulacions equivalents de la completesa; en aquest quadern
hem demostrat les implicacions indicades, no tots els recíprocs.

## 5. Tornem al problema inicial
L'algorisme d'Arquímedes proporciona intervals encaixats d'amplada que tendeix a zero.
La completesa garanteix un únic real comú. La geometria identifica aquest real amb π.
El quadern 01 mostra que els intervals racionals poden no contenir cap racional comú.

## Activitats
1. Dona un conjunt que tingui suprem però no màxim, diferent de l'exemple inicial.
2. Per què cal que els intervals siguin tancats? Justifica el cas $(0,1/n)$.
3. Explica la funció de cadascuna d'aquestes idees: encaixament, amplada i completesa.
4. Escriu cinc frases que connectin Arquímedes, Dedekind i Cauchy sense dir que
   un càlcul numèric finit demostra la completesa.

<details><summary>Pistes</summary>

$(0,1)$ té suprem 1 i no màxim. Si x>0, existeix n amb $1/n<x$, de manera que
x no és a tots els $(0,1/n)$; 0 tampoc hi pertany. L'encaixament i la completesa
proporcionen existència en el cas tancat; l'amplada que tendeix a zero proporciona unicitat.
</details>


**Navegació:** [Anterior](05_Successions_Cauchy.ipynb) · [Índex i guia](README.md) · [Següent](07_Limits_Continuitat.ipynb)
